# 무기한 선물 펀딩비 (Binance 공개 API)
- 대상: `tradable_assets.md`의 **코인 무기한 선물** 6개 (USD-M 4 + COIN-M 2)
- 엔드포인트: USD-M `fapi/v1/fundingRate`, COIN-M `dapi/v1/fundingRate` (1회 최대 1000건, API 키 불필요)
- 펀딩비 부호: **양수 = 롱이 숏에게 지급**, 음수 = 숏이 롱에게 지급
- 지급액 = 포지션 명목가치 × fundingRate (보통 8시간마다 UTC 00·08·16시, 시기에 따라 주기가 바뀔 수 있음)
- 일별 합계는 UTC 날짜 기준 → `crypto_perp_close.csv`(UTC 일봉)와 날짜가 맞음

In [ ]:
import os
import re, time
from pathlib import Path
import pandas as pd
import requests

PROJECT_DIR = Path(os.environ.get("FP2_PROJECT_DIR", Path.cwd()))   # 데이터·결과 폴더 (환경변수 FP2_PROJECT_DIR로 지정, 기본=노트북 폴더)
MD_PATH = PROJECT_DIR / "tradable_assets.md"
START_MS = int(pd.Timestamp("2017-01-01", tz="UTC").timestamp() * 1000)   # 바이낸스 선물 출시 전
NOW_MS = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)

## 1. md 파일에서 무기한 선물 심볼 추출

In [ ]:
text = MD_PATH.read_text(encoding="utf-8")
fut_sec = re.search(r"^## 코인 무기한 선물.*?$(.*?)(?=^## |\Z)", text, flags=re.M | re.S).group(1)
usdm = re.findall(r"\b([A-Z]+USDT)\.P\b", fut_sec)
coinm = re.findall(r"\b([A-Z]+USD_PERP)\b", fut_sec)
jobs = [(s, "https://fapi.binance.com/fapi/v1/fundingRate", f"{s}.P") for s in usdm] + \
       [(s, "https://dapi.binance.com/dapi/v1/fundingRate", s) for s in coinm]
[j[2] for j in jobs]

## 2. 다운로드 (페이지네이션)

In [ ]:
def get(url, params, retries=5):
    for i in range(retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (418, 429):
            time.sleep(int(r.headers.get("Retry-After", 2 ** i)))
        else:
            r.raise_for_status()
    raise RuntimeError(f"failed: {url} {params}")

def fetch_funding(symbol, url):
    rows, start = [], START_MS
    while start < NOW_MS:
        data = get(url, dict(symbol=symbol, startTime=start, limit=1000))
        if not data:
            break
        rows += data
        start = data[-1]["fundingTime"] + 1
        if len(data) < 1000:
            break
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    df["time"] = pd.to_datetime(df["fundingTime"], unit="ms").dt.floor("s")   # 끝자리 몇 ms 오차 제거
    df["fundingRate"] = df["fundingRate"].astype(float)
    df["markPrice"] = pd.to_numeric(df["markPrice"], errors="coerce")       # 초기 구간은 빈칸
    return df.drop_duplicates("fundingTime")[["time", "fundingRate", "markPrice"]]

funding = {}
for sym, url, label in jobs:
    funding[label] = fetch_funding(sym, url)
    d = funding[label]
    print(f"{label:12s} {d.time.min()} ~ {d.time.max()}  {len(d):5d}건")

## 3. 펀딩 주기 확인 (8시간 외 주기가 있었는지)

In [ ]:
for label, d in funding.items():
    gap_h = d["time"].diff().dt.total_seconds().div(3600).round().value_counts().sort_index()
    print(label, dict(gap_h.astype(int)))

## 4. 정리: 원본(long) + 일별 합계(wide)

In [ ]:
events = pd.concat(funding, names=["symbol"]).reset_index(level=0).reset_index(drop=True)

# 오늘(UTC) 펀딩은 아직 하루치가 다 안 쌓였으므로 일별 합계에서 제외 (일봉도 오늘 캔들 제외)
today = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
done = events[events["time"] < today]

daily = (done.assign(date=events["time"].dt.normalize())
               .pivot_table(index="date", columns="symbol", values="fundingRate", aggfunc="sum")
               [[j[2] for j in jobs]])
n_per_day = (done.assign(date=events["time"].dt.normalize())
                   .pivot_table(index="date", columns="symbol", values="fundingRate", aggfunc="count"))

summary = pd.DataFrame({
    "first": events.groupby("symbol")["time"].min(),
    "last": events.groupby("symbol")["time"].max(),
    "n_events": events.groupby("symbol").size(),
    "mean_rate_8h": events.groupby("symbol")["fundingRate"].mean(),
    "max_rate": events.groupby("symbol")["fundingRate"].max(),
    "min_rate": events.groupby("symbol")["fundingRate"].min(),
    "annualized_mean": daily.mean() * 365,          # 롱 보유 시 연간 평균 비용
}).loc[[j[2] for j in jobs]]
summary

## 5. 검증: 일봉 데이터와 날짜 정합성

In [ ]:
perp_close = pd.read_csv(PROJECT_DIR / "crypto_perp_close.csv", index_col=0, parse_dates=True)
for c in daily.columns:
    px_days = perp_close[c].dropna().index
    fd_days = daily[c].dropna().index
    print(f"{c:12s} 일봉에 있는데 펀딩 없는 날: {len(px_days.difference(fd_days))}, "
          f"하루 펀딩 횟수 분포: {dict(n_per_day[c].dropna().astype(int).value_counts())}")

## 6. 저장

In [ ]:
events.to_csv(PROJECT_DIR / "crypto_funding_events.csv", index=False)   # 펀딩 시점별 원본
daily.to_csv(PROJECT_DIR / "crypto_funding_daily.csv")                  # UTC 일별 합계
summary.to_csv(PROJECT_DIR / "crypto_funding_summary.csv")
print("saved:", events.shape, daily.shape)